# Marine Oil Spill Segmentation — U-Net (Focal Loss + IoU)

**Architecture:** Depth U-Net with Focal Loss and IoU metric  
**Input:** Sentinel-1 dual-polarization SAR images (VV + VH, 512×512)  
**Output:** Binary segmentation mask (Oil=1, No Oil=0)  

## 1. Imports

In [ ]:
import os
import json
import random
import glob
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import rasterio

from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv2D, MaxPooling2D, UpSampling2D,
    concatenate, BatchNormalization, Activation, Dropout
)
from tensorflow.keras.callbacks import ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split

print('TensorFlow version:', tf.__version__)

## 2. Configuration

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Paths ─────────────────────────────────────────────────────────────────────
# Update BASE_PATH to point to your dataset root.
# Expected structure:
#   BASE_PATH/
#     Train/Images/Oil/        <- dual-pol .tif files
#     Train/Images/No_Oil/
#     Train/Images/Lookalike/
#     Train/Masks/Oil/         <- binary .tif masks (0=background, 1=oil)
#     Test/Images/Oil/
#     Test/Images/No_Oil/
#     Test/Images/Lookalike/
#     Test/Masks/Oil/
BASE_PATH = r"C:\FinalYear\Dataset-OG"

TRAIN_IMG_DIR  = os.path.join(BASE_PATH, 'Train', 'Images')
TRAIN_MASK_DIR = os.path.join(BASE_PATH, 'Train', 'Masks')
TEST_IMG_DIR   = os.path.join(BASE_PATH, 'Test',  'Images')
TEST_MASK_DIR  = os.path.join(BASE_PATH, 'Test',  'Masks')

# Path to your saved CNN classification model (from Final_Classification.ipynb)
CNN_MODEL_PATH = "best_model.h5"

# ── Checkpointing ─────────────────────────────────────────────────────────────
CKPT_DIR     = "unet_checkpoints"   # folder for all per-epoch .h5 files
HISTORY_FILE = "unet_full_history.json"  # accumulates metrics across all runs
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE       = (512, 512)   # paper uses 512x512
BATCH_SIZE     = 4            # lower than classification due to larger model
EPOCHS_PER_RUN = 10           # train 10 epochs at a time; resume as many times as needed
LR             = 1e-4

# Focal Loss parameters (paper: Lin et al., 2017)
FOCAL_ALPHA = 0.25   # weight for the positive (oil) class
FOCAL_GAMMA = 2.0    # focusing parameter; 0 = standard cross-entropy

# U-Net filter schedule (best config from paper Table 3)
FILTERS = [16, 32, 64, 128, 256, 512, 1024]

## 3. Custom Metrics and Loss
The paper found that **accuracy and cross-entropy are biased by class imbalance** in segmentation.
Instead it uses **Focal Loss** (loss) and **IoU** (metric).

In [ ]:
# ── IoU metric ────────────────────────────────────────────────────────────────
def iou_metric(y_true, y_pred, smooth=1e-6):
    '''Intersection Over Union — the primary metric used in the paper.'''
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    y_true = tf.cast(y_true, tf.float32)
    intersection = K.sum(y_true * y_pred, axis=[1, 2, 3])
    union = K.sum(y_true, axis=[1, 2, 3]) + K.sum(y_pred, axis=[1, 2, 3]) - intersection
    return K.mean((intersection + smooth) / (union + smooth))


# ── Focal Loss ────────────────────────────────────────────────────────────────
def focal_loss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
    '''
    Binary Focal Loss (Lin et al., 2017).
    Focuses training on hard pixels (the sparse oil class).
    Modulation factor: (1 - p_t)^gamma
    '''
    def loss_fn(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        bce_pos = -y_true * tf.math.log(y_pred)
        bce_neg = -(1.0 - y_true) * tf.math.log(1.0 - y_pred)
        focal_pos = alpha       * tf.pow(1.0 - y_pred, gamma) * bce_pos
        focal_neg = (1 - alpha) * tf.pow(y_pred, gamma)       * bce_neg
        return tf.reduce_mean(focal_pos + focal_neg)
    return loss_fn


# ── Dice coefficient (supplementary metric) ───────────────────────────────────
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_pred = tf.cast(y_pred > 0.5, tf.float32)
    y_true = tf.cast(y_true, tf.float32)
    intersection = K.sum(y_true * y_pred, axis=[1, 2, 3])
    return K.mean((2.0 * intersection + smooth) /
                  (K.sum(y_true, axis=[1, 2, 3]) + K.sum(y_pred, axis=[1, 2, 3]) + smooth))

print('Custom loss and metrics defined.')

## 4. U-Net Architecture
Faithful to the paper's best configuration:
- Filters: 16 -> 32 -> 64 -> 128 -> 256 -> 512 -> 1024
- **Upsampling2D** in expansion (not Conv2DTranspose — paper found this better)
- Kernel 3x3, ReLU activations, sigmoid output
- BatchNormalization added for training stability

In [ ]:
def conv_block(x, filters, dropout_rate=0.0):
    '''Two Conv2D -> BN -> ReLU layers (standard U-Net building block).'''
    x = Conv2D(filters, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    x = Conv2D(filters, (3, 3), padding='same', kernel_initializer='he_normal')(x)
    x = BatchNormalization()(x)
    x = Activation('relu')(x)
    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)
    return x


def build_unet(input_shape=(512, 512, 2), filters=None):
    '''
    Depth U-Net matching the paper best configuration (Table 3).
    input_shape: (H, W, C) — 2 channels for VV and VH polarizations.
    filters:     list of filter sizes for each encoder level.
    '''
    if filters is None:
        filters = FILTERS   # [16, 32, 64, 128, 256, 512, 1024]

    inputs = Input(shape=input_shape, name='sar_input')

    # Encoder (contraction path)
    skips = []
    x = inputs
    for i, f in enumerate(filters[:-1]):
        dr = 0.1 if i >= len(filters) - 3 else 0
        x = conv_block(x, f, dropout_rate=dr)
        skips.append(x)
        x = MaxPooling2D((2, 2))(x)

    # Bottleneck
    x = conv_block(x, filters[-1], dropout_rate=0.2)

    # Decoder (expansion path)
    # Paper uses Upsampling2D — key finding of the study
    for f, skip in zip(reversed(filters[:-1]), reversed(skips)):
        x = UpSampling2D((2, 2))(x)
        x = concatenate([x, skip])
        x = conv_block(x, f)

    # Output: 1x1 conv -> sigmoid
    outputs = Conv2D(1, (1, 1), activation='sigmoid', name='segmentation_map')(x)

    model = Model(inputs=inputs, outputs=outputs, name='Depth_UNet')
    return model


# Quick check
tmp = build_unet()
tmp.summary(line_length=100)
del tmp

## 5. Data Loading

In [ ]:
def load_sar_image(path):
    '''
    Load a dual-pol Sentinel-1 SAR GeoTIFF and return a normalised
    (512, 512, 2) float32 array. Clipping values match the classification notebook.
    '''
    with rasterio.open(path) as src:
        vv = src.read(1).astype(np.float32)
        vh = src.read(2).astype(np.float32)
    vv = np.clip(vv, -35, 5)
    vh = np.clip(vh, -40, 0)
    vv = (vv + 35) / 40.0
    vh = (vh + 40) / 40.0
    img = np.stack([vv, vh], axis=-1)
    img = tf.image.resize(img, IMG_SIZE).numpy()
    return img


def load_mask(path):
    '''
    Load a binary segmentation mask GeoTIFF.
    Returns (512, 512, 1) float32 with values in {0, 1}.
    '''
    with rasterio.open(path) as src:
        mask = src.read(1).astype(np.float32)
    mask = (mask > 0).astype(np.float32)
    mask = tf.image.resize(mask[..., np.newaxis], IMG_SIZE, method='nearest').numpy()
    return mask


def build_segmentation_dataset(images_root, masks_root):
    oil_img_dir  = os.path.join(images_root, 'Oil')
    oil_mask_dir = os.path.join(masks_root,  'Oil')
    img_files  = sorted(glob.glob(os.path.join(oil_img_dir,  '*.tif')))
    mask_files = sorted(glob.glob(os.path.join(oil_mask_dir, '*.tif')))
    assert len(img_files) == len(mask_files), (
        f'Mismatch: {len(img_files)} images vs {len(mask_files)} masks'
    )
    print(f'Found {len(img_files)} Oil image-mask pairs.')
    return np.array(img_files), np.array(mask_files)


def data_generator(img_paths, mask_paths, batch_size, augment=False, shuffle=True):
    '''Infinite generator yielding (batch_images, batch_masks).'''
    n = len(img_paths)
    indices = np.arange(n)
    while True:
        if shuffle:
            np.random.shuffle(indices)
        for start in range(0, n, batch_size):
            batch_idx = indices[start:start + batch_size]
            imgs, masks = [], []
            for i in batch_idx:
                img  = load_sar_image(img_paths[i])
                mask = load_mask(mask_paths[i])
                if augment:
                    if random.random() > 0.5:
                        img  = np.fliplr(img);  mask = np.fliplr(mask)
                    if random.random() > 0.5:
                        img  = np.flipud(img);  mask = np.flipud(mask)
                    k = random.choice([0, 1, 2, 3])
                    img  = np.rot90(img,  k)
                    mask = np.rot90(mask, k)
                imgs.append(img)
                masks.append(mask)
            yield np.array(imgs, dtype=np.float32), np.array(masks, dtype=np.float32)

print('Data loading utilities defined.')

## 6. Prepare Train / Validation Split

In [ ]:
all_img_paths, all_mask_paths = build_segmentation_dataset(TRAIN_IMG_DIR, TRAIN_MASK_DIR)

# 66% train / 34% validation — matches paper Section 2.1
train_imgs, val_imgs, train_masks, val_masks = train_test_split(
    all_img_paths, all_mask_paths, test_size=0.34, random_state=SEED
)

print(f'Train samples : {len(train_imgs)}')
print(f'Val   samples : {len(val_imgs)}')

train_steps = max(1, len(train_imgs) // BATCH_SIZE)
val_steps   = max(1, len(val_imgs)   // BATCH_SIZE)

train_gen = data_generator(train_imgs, train_masks, BATCH_SIZE, augment=True,  shuffle=True)
val_gen   = data_generator(val_imgs,   val_masks,   BATCH_SIZE, augment=False, shuffle=False)

## 7. History Helpers
These functions persist training metrics to disk so they survive kernel restarts.
They also tell us the global epoch offset needed for correct checkpoint filenames.

In [ ]:
def load_full_history():
    '''Load accumulated history dict from disk (empty dict if first run).'''
    if os.path.exists(HISTORY_FILE):
        with open(HISTORY_FILE) as f:
            return json.load(f)
    return {}


def append_and_save_history(full_history, new_history_obj):
    '''Append this run History.history into the cumulative JSON file.'''
    for key, values in new_history_obj.history.items():
        full_history.setdefault(key, []).extend(values)
    with open(HISTORY_FILE, 'w') as f:
        json.dump(full_history, f, indent=2)
    total = len(full_history.get('loss', []))
    print(f'History saved -> {HISTORY_FILE}  (total epochs so far: {total})')
    return full_history


def get_epoch_offset():
    '''Return number of epochs already completed (reads history file).'''
    return len(load_full_history().get('loss', []))


def get_latest_checkpoint():
    '''Return path of the highest-numbered epoch checkpoint, or None.'''
    ckpts = sorted(glob.glob(os.path.join(CKPT_DIR, 'unet_epoch_*.h5')))
    return ckpts[-1] if ckpts else None


print('History helpers defined.')
print(f'Epochs completed so far: {get_epoch_offset()}')
latest = get_latest_checkpoint()
print(f'Latest checkpoint      : {latest if latest else "none — fresh start"}')

## 8. EpochCheckpoint Callback
Saves the full model after every epoch using the **global** epoch number,
so filenames are unique across resumed runs:
```
unet_checkpoints/unet_epoch_001.h5
unet_checkpoints/unet_epoch_002.h5
...  (run 1 ends)
unet_checkpoints/unet_epoch_011.h5   <- resume run 2
...
```

In [ ]:
class EpochCheckpoint(tf.keras.callbacks.Callback):
    '''
    Saves the model after every epoch using the global epoch number.
    Pass epoch_offset = total epochs already completed before this run.
    '''
    def __init__(self, ckpt_dir, epoch_offset=0):
        super().__init__()
        self.ckpt_dir     = ckpt_dir
        self.epoch_offset = epoch_offset

    def on_epoch_end(self, epoch, logs=None):
        global_epoch = self.epoch_offset + epoch + 1   # 1-based
        path = os.path.join(self.ckpt_dir, f'unet_epoch_{global_epoch:03d}.h5')
        self.model.save(path)
        iou = logs.get('val_iou_metric', float('nan'))
        print(f'  -> Saved unet_epoch_{global_epoch:03d}.h5   val_IoU={iou:.4f}')


# ReduceLROnPlateau works correctly across runs because optimizer
# state (including lr) is saved inside the .h5 checkpoint.
reduce_lr = ReduceLROnPlateau(
    monitor='val_iou_metric', mode='max',
    factor=0.5, patience=5, min_lr=1e-7, verbose=1
)

print('EpochCheckpoint callback defined.')

## 9. Train — First Run (Epochs 1–10)
Run this cell **once** from scratch.  
Saves `unet_epoch_001.h5` … `unet_epoch_010.h5` **and** a final `unet_final_epoch010.h5`.

In [ ]:
# ── Determine starting point ─────────────────────────────────────────────────
epoch_offset = get_epoch_offset()        # 0 on a clean start
latest_ckpt  = get_latest_checkpoint()   # None on a clean start

if latest_ckpt:
    print(f'Resuming from checkpoint: {latest_ckpt}')
    unet_model = tf.keras.models.load_model(
        latest_ckpt,
        custom_objects={'loss_fn': focal_loss(), 'iou_metric': iou_metric, 'dice_coef': dice_coef}
    )
else:
    print('Starting training from scratch.')
    unet_model = build_unet()
    unet_model.compile(
        optimizer=Adam(learning_rate=LR),
        loss=focal_loss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA),
        metrics=[iou_metric, dice_coef, 'accuracy']
    )

print(f'Epoch offset : {epoch_offset}  '
      f'(next checkpoint will be epoch_{epoch_offset+1:03d})')

# ── Build callbacks for this run ─────────────────────────────────────────────
run_callbacks = [
    EpochCheckpoint(CKPT_DIR, epoch_offset=epoch_offset),
    reduce_lr,
]

# ── Train for EPOCHS_PER_RUN epochs ─────────────────────────────────────────
history_run = unet_model.fit(
    train_gen,
    steps_per_epoch=train_steps,
    epochs=EPOCHS_PER_RUN,
    validation_data=val_gen,
    validation_steps=val_steps,
    callbacks=run_callbacks,
    verbose=1
)

# ── Save final model for this run ────────────────────────────────────────────
final_epoch = epoch_offset + EPOCHS_PER_RUN
final_path  = os.path.join(CKPT_DIR, f'unet_final_epoch{final_epoch:03d}.h5')
unet_model.save(final_path)
print(f'\nFinal model for this run saved -> {final_path}')

# ── Accumulate history ───────────────────────────────────────────────────────
full_history = append_and_save_history(load_full_history(), history_run)
print(f'\nDone. To continue training, run the Resume cell below.')

## 10. Resume Training — Re-run This Cell to Add 10 More Epochs
Each time you run this cell it:
1. Reads `unet_full_history.json` to find how many epochs are already done
2. Auto-loads the latest checkpoint (`unet_epoch_NNN.h5`)
3. Trains for the next `EPOCHS_PER_RUN` (10) epochs
4. Saves every epoch + a new `unet_final_epochNNN.h5`
5. Appends metrics to the cumulative history file

**Just keep re-running this single cell** — no edits needed.

In [ ]:
# ── Auto-detect where we left off ───────────────────────────────────────────
epoch_offset = get_epoch_offset()
latest_ckpt  = get_latest_checkpoint()

if latest_ckpt is None:
    raise FileNotFoundError(
        'No checkpoint found. Run the First Run cell (Section 9) first.'
    )

print(f'Epochs completed so far  : {epoch_offset}')
print(f'Loading checkpoint       : {latest_ckpt}')
print(f'Will train epochs        : {epoch_offset+1} -> {epoch_offset+EPOCHS_PER_RUN}')

unet_model = tf.keras.models.load_model(
    latest_ckpt,
    custom_objects={'loss_fn': focal_loss(), 'iou_metric': iou_metric, 'dice_coef': dice_coef}
)

# ── Callbacks ────────────────────────────────────────────────────────────────
run_callbacks = [
    EpochCheckpoint(CKPT_DIR, epoch_offset=epoch_offset),
    reduce_lr,
]

# ── Train ────────────────────────────────────────────────────────────────────
history_run = unet_model.fit(
    train_gen,
    steps_per_epoch=train_steps,
    epochs=EPOCHS_PER_RUN,
    validation_data=val_gen,
    validation_steps=val_steps,
    callbacks=run_callbacks,
    verbose=1
)

# ── Save final model for this run ────────────────────────────────────────────
final_epoch = epoch_offset + EPOCHS_PER_RUN
final_path  = os.path.join(CKPT_DIR, f'unet_final_epoch{final_epoch:03d}.h5')
unet_model.save(final_path)
print(f'\nFinal model saved -> {final_path}')

# ── Accumulate history ───────────────────────────────────────────────────────
full_history = append_and_save_history(load_full_history(), history_run)
print(f'Total epochs trained so far: {len(full_history["loss"])}')
print('Re-run this cell to train the next 10 epochs.')

## 11. Plot Full Training History
Reads the cumulative JSON file — shows the complete curve across all runs.

In [ ]:
full_history = load_full_history()

if not full_history:
    print('No history found yet — train at least one run first.')
else:
    epochs_range = range(1, len(full_history['loss']) + 1)
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].plot(epochs_range, full_history['iou_metric'],     label='Train IoU',  color='red')
    axes[0].plot(epochs_range, full_history['val_iou_metric'], label='Val IoU',    color='green')
    axes[0].set_title('IoU (Primary Metric)')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('IoU')
    axes[0].legend(); axes[0].grid(True)

    axes[1].plot(epochs_range, full_history['loss'],     label='Train Focal Loss', color='red')
    axes[1].plot(epochs_range, full_history['val_loss'], label='Val Focal Loss',   color='green')
    axes[1].set_title('Focal Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
    axes[1].legend(); axes[1].grid(True)

    axes[2].plot(epochs_range, full_history['accuracy'],     label='Train Acc', color='red')
    axes[2].plot(epochs_range, full_history['val_accuracy'], label='Val Acc',   color='green')
    axes[2].set_title('Pixel Accuracy (secondary)')
    axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Accuracy')
    axes[2].legend(); axes[2].grid(True)

    plt.suptitle(f'U-Net Training History ({len(full_history["loss"])} epochs total)', fontsize=14)
    plt.tight_layout()
    plt.savefig('unet_training_history.png', dpi=150)
    plt.show()
    print('Plot saved to unet_training_history.png')

## 12. Evaluation on Test Set
Load any checkpoint (or the latest) and evaluate.

In [ ]:
from tensorflow.keras.models import load_model

# Load the latest checkpoint (or specify a path manually)
eval_ckpt = get_latest_checkpoint()
print(f'Evaluating: {eval_ckpt}')

best_unet = load_model(
    eval_ckpt,
    custom_objects={'loss_fn': focal_loss(), 'iou_metric': iou_metric, 'dice_coef': dice_coef}
)


def evaluate_on_set(img_paths, mask_paths, model, threshold=0.5):
    ious, accs = [], []
    for ip, mp in zip(img_paths, mask_paths):
        img      = load_sar_image(ip)[np.newaxis]
        mask     = load_mask(mp)
        pred     = model.predict(img, verbose=0)[0]
        pred_bin = (pred > threshold).astype(np.float32)
        inter = np.sum(mask * pred_bin)
        union = np.sum(mask) + np.sum(pred_bin) - inter
        ious.append((inter + 1e-6) / (union + 1e-6))
        accs.append(np.mean(mask == pred_bin))
    return np.mean(ious), np.mean(accs)


test_img_paths, test_mask_paths = build_segmentation_dataset(TEST_IMG_DIR, TEST_MASK_DIR)
mean_iou, mean_acc = evaluate_on_set(test_img_paths, test_mask_paths, best_unet)
print(f'\nTest Results:')
print(f'  Mean IoU            : {mean_iou*100:.2f} %')
print(f'  Mean Pixel Accuracy : {mean_acc*100:.2f} %')

## 13. Visualise Predictions

In [ ]:
def visualise_predictions(img_paths, mask_paths, model, n=4, threshold=0.5):
    n = min(n, len(img_paths))
    fig, axes = plt.subplots(n, 3, figsize=(12, 4 * n))
    if n == 1: axes = axes[np.newaxis]
    for row, (ip, mp) in enumerate(zip(img_paths[:n], mask_paths[:n])):
        img  = load_sar_image(ip)
        mask = load_mask(mp)
        pred = model.predict(img[np.newaxis], verbose=0)[0]
        pred_bin = (pred > threshold).astype(np.float32)
        inter = np.sum(mask * pred_bin)
        union = np.sum(mask) + np.sum(pred_bin) - inter
        iou  = (inter + 1e-6) / (union + 1e-6)
        acc  = np.mean(mask == pred_bin)
        axes[row, 0].imshow(img[:, :, 0], cmap='gray')
        axes[row, 0].set_title('SAR Image (VV)'); axes[row, 0].axis('off')
        axes[row, 1].imshow(mask[:, :, 0], cmap='gray')
        axes[row, 1].set_title('Ground Truth'); axes[row, 1].axis('off')
        axes[row, 2].imshow(pred_bin[:, :, 0], cmap='gray')
        axes[row, 2].set_title(f'Prediction  IoU={iou*100:.1f}%  Acc={acc*100:.1f}%')
        axes[row, 2].axis('off')
    plt.suptitle('U-Net Segmentation Results', fontsize=14)
    plt.tight_layout()
    plt.savefig('unet_predictions.png', dpi=150)
    plt.show()

visualise_predictions(test_img_paths, test_mask_paths, best_unet, n=4)

## 14. Two-Stage Pipeline: CNN -> U-Net
Full pipeline from the paper (Section 4.4 / Fig. S18):
1. CNN classifier filters out No-Oil & look-alike tiles
2. U-Net segments oil within confirmed Oil tiles only

In [ ]:
from tensorflow.keras.models import load_model as keras_load

cnn_model = keras_load(CNN_MODEL_PATH)
print(f'CNN classifier loaded from: {CNN_MODEL_PATH}')


def two_stage_predict(img_path, cnn, unet, cnn_threshold=0.5, seg_threshold=0.5):
    img = load_sar_image(img_path)
    cnn_prob = float(cnn.predict(img[np.newaxis], verbose=0)[0][0])
    is_oil = cnn_prob > cnn_threshold
    if not is_oil:
        return cnn_prob, False, None
    raw_pred = unet.predict(img[np.newaxis], verbose=0)[0]
    seg_mask = (raw_pred > seg_threshold).astype(np.float32)
    return cnn_prob, True, seg_mask


def run_pipeline_on_folder(img_dir, cnn, unet, label=None):
    files = sorted(glob.glob(os.path.join(img_dir, '*.tif')))
    print(f'\nRunning pipeline on {len(files)} images ({label or img_dir})')
    detected = 0
    for fp in files:
        prob, is_oil, mask = two_stage_predict(fp, cnn, unet)
        if is_oil:
            detected += 1
            print(f'  [OIL]    {os.path.basename(fp):40s}  p={prob:.3f}  oil_pixels={int(mask.sum())}')
        else:
            print(f'  [NO OIL] {os.path.basename(fp):40s}  p={prob:.3f}')
    print(f'\nDetected oil in {detected}/{len(files)} images.')


run_pipeline_on_folder(os.path.join(TEST_IMG_DIR, 'Oil'), cnn_model, best_unet, label='Test-Oil')

## 15. Spill Area Estimation
Optional post-processing step from the paper (Section 4.4 / Fig. S18, Step 6).

In [ ]:
def estimate_spill_area(seg_mask, pixel_size_m=10.0):
    '''
    Estimate oil spill area in km2 from a binary segmentation mask.
    Sentinel-1 GRD resolution: 10 m/pixel (paper Section 2.1).
    '''
    if seg_mask.ndim == 3:
        seg_mask = seg_mask[:, :, 0]
    oil_pixels = int(seg_mask.sum())
    area_km2   = oil_pixels * (pixel_size_m ** 2) / 1e6
    return area_km2


sample_ip = test_img_paths[0]
_, is_oil, seg_mask = two_stage_predict(sample_ip, cnn_model, best_unet)

if is_oil and seg_mask is not None:
    area = estimate_spill_area(seg_mask)
    print(f'Estimated spill area: {area:.4f} km2')
    print(f'Oil pixels          : {int(seg_mask.sum())}')
else:
    print('No oil detected — area estimation skipped.')